# Projeto ECAC

## 2. Data understanding

### 2.1 What is each column supposed to contain? Which are measurements, which are identifiers, which are administrative?

| Nome da coluna | Descrição | Tipo de dados |
|:--|:--|:--|
| Audio_Song | id do excerto de áudio, único por linha | identificador |
| Lyric_Song | id do ficheiro de letra | identificador |
| AllMusic Id	| id da faixa no catálogo AllMusic | identificador |
| Artist | artista, com uma só grafia por artista	| identificador |
| Title | título da faixa	| identificador |
| AppearancesTrackIDs	| ids de outras aparições da mesma gravação | identificador |
| AppearancesAlbumIDs	| id de álbum para cada aparição, alinhado com a anterior | identificador |
| Duration | duração da faixa completa, em segundos, > 0	| medida |
| Year, LowestYear, ActualYear | datas de lançamento, em três formatos diferentes; LowestYear ≤ ActualYear, e ActualYear não pode estar no futuro | medida |
| Moods, MoodsAll, Genres, Themes, Styles	| etiquetas do AllMusic (listas separadas por vírgulas) | medida |
| MoodsAllWeights, GenreWeights, ThemeWeights, StyleWeights | um peso por etiqueta, na mesma ordem e quantidade | medida |
| num_Genres, num_MoodsAll | contagem de etiquetas, que deve bater com as listas | medida |
| Relevance | score que o AllMusic devolveu para a query que recuperou a faixa, > 0 | medida |
| AllMusic Extraction Date | qual das duas recolhas produziu o registo (Old/New) | administrativa |
| Sample | se havia amostra de 30 s na altura da recolha | administrativa |
| SampleURL | endereço de onde o excerto foi obtido | administrativa |
| Quadrant | alvo: Q1 a Q4 | alvo |
| Arousal, Valence | alvo em forma contínua, entre 0 e 1 | alvo |

### 2.2 Do the four sources join one to one? If not, where does it break, and what follows from that?

In [12]:
from pathlib import Path
import pandas as pd

MEDIA  = Path(r"C:\Users\simao\Documents\3º Uni\ECAC\dataset\dataset")
TABLES = Path(r"C:\Users\simao\Documents\3º Uni\ECAC\dataset\build\merge_ecac\merge_ecac")

meta = pd.read_csv(TABLES / "metadata.csv")

def ids(pasta, ext):
    return {p.stem for p in (MEDIA / pasta).rglob(f"*.{ext}")}

fontes = {
    "audio":  (ids("audio", "mp3"), set(meta["Audio_Song"])),
    "lyrics": (ids("lyrics", "txt"), set(meta["Lyric_Song"])),
    "covers": (ids("album_covers", "jpg"), set(meta["Audio_Song"])),
}

print("linhas:", len(meta))
print("distintas:", meta["Audio_Song"].nunique()) 
print("repetidas:", meta["Audio_Song"].duplicated().sum())

for nome, (ficheiros, tabela) in fontes.items():
    print(nome, "| tabela sem ficheiro:", len(tabela - ficheiros), "| ficheiro sem tabela:", len(ficheiros - tabela))

linhas: 1348
distintas: 1323
repetidas: 25
audio | tabela sem ficheiro: 0 | ficheiro sem tabela: 893
lyrics | tabela sem ficheiro: 0 | ficheiro sem tabela: 893
covers | tabela sem ficheiro: 109 | ficheiro sem tabela: 805


Não. Falha em três sítios:

- Linhas repetidas na tabela. São 1348 linhas, mas só 1323 músicas distintas: 25 músicas aparecem em duas linhas.
- Áudio e letras sem linha. Todas as linhas da tabela têm áudio e letra, mas as pastas contêm 2216 ficheiros de cada tipo e a tabela só descreve 1323. Ficam 893 áudios e 893 letras sem metadados nem quadrante.
- Capas incompletas. Há 2019 capas: 109 músicas da tabela não têm capa e 805 capas não correspondem a nenhuma linha.

### 2.3 From the documentation alone, before computing anything: which columns are suspected of giving away the target? Record the list.

Moods, MoodsAll, MoodsAllWeights e num_MoodsAll são suspeitas de target leakage, pois o enunciado refere que os rótulos emocionais foram obtidos a partir de emotion tags do AllMusic. 

O caminho dos ficheiros de média (`audio/<Quadrant>/`) também revela o alvo, porque há uma pasta por quadrante. Não é coluna, mas não pode ser usado como atributo.

Themes e ThemeWeights são consideradas hipóteses a investigar, por pertencerem à mesma família de tags editoriais.

## 4. Cleaning

### 4.1 Duplicates

#### 4.1.1 Identify the identical rows.

In [13]:
print("linhas idênticas a uma anterior:", meta.duplicated().sum())
print("linhas envolvidas:", meta.duplicated(keep=False).sum())
print("ids repetidos:", meta["Audio_Song"].duplicated().sum())

linhas idênticas a uma anterior: 25
linhas envolvidas: 50
ids repetidos: 25


Foram identificadas 25 linhas duplicadas exatas. No total, 50 linhas estão envolvidas nestas duplicações, formando 25 pares de registos idênticos. Verificou-se ainda a existência de 25 valores repetidos de Audio_Song, correspondentes precisamente a esses pares duplicados. Como não acrescentam informação nova, estes registos podem ser removidos sem perda de informação.

#### 4.1.2 No column states which rows are the same recording. Compare four kinds of evidence: the identifier columns, the artist and title, the lyrics, the audio. Do they agree? Which of them establishes that two rows are the same recording?

In [14]:
import hashlib

u = meta.drop_duplicates().copy()

audio_f  = {p.stem: p for p in (MEDIA / "audio").rglob("*.mp3")}
lyrics_f = {p.stem: p for p in (MEDIA / "lyrics").rglob("*.txt")}

u["art_tit"] = (u["Artist"].str.lower().str.strip() + "|" + u["Title"].str.lower().str.strip())
u["audio"] = [hashlib.md5(audio_f[i].read_bytes()).hexdigest() for i in u["Audio_Song"]]
u["letra"] = [lyrics_f[i].read_text(encoding="utf-8", errors="ignore").strip() for i in u["Lyric_Song"]]

for i in ["AllMusic Id", "art_tit", "letra", "audio"]:
    print(i, "->", u[i].dropna().duplicated().sum())

for j in ["art_tit", "letra", "audio"]:
    print(f"\n {j} ")
    print(u.loc[u[j].duplicated(keep=False), ["Artist", "Title", "Audio_Song", "Lyric_Song"]])

AllMusic Id -> 3
art_tit -> 1
letra -> 6
audio -> 2

 art_tit 
       Artist           Title    Audio_Song    Lyric_Song
67    Prince   U Got the Look  MT0004444570  MT0004444570
1160  Prince   U GOT THE LOOK          A005          L055

 letra 
                              Artist                         Title  \
67                           Prince                 U Got the Look   
83                The Isley Brothers               Twist and Shout   
155                    Hank Williams               I'm So Lonesome   
165                         Godsmack            I F****** Hate You   
521   Hank Ballard & the Midnighters         Look at Little Sister   
663                           Prince             Kiss - Remastered   
679                      The Beatles               Twist and Shout   
998                   Hank  Williams  I'm So Lonesome, I Could Cry   
1058                        Godsmack            I Fucking Hate You   
1118                    Hank Ballard         Look at L

As quatro provas não concordam. Depois de remover as cópias exatas, o áudio idêntico apanha 2 pares, a letra 6 e o artista e título 1. O `AllMusic Id` não conta como prova: os seus 3 repetidos são o valor not_found em 4 músicas sem relação entre si, uma sentinela e não um identificador. 

Cada prova mede uma coisa diferente. A letra identifica a canção e não a gravação ("Twist and Shout" tem a mesma letra nos Isley Brothers e nos Beatles). O artista e título falham quando o título muda ("I F****** Hate You" e "I Fucking Hate You"). O áudio é a prova que estabelece que duas linhas são a mesma gravação. Mas só apanha cópias idênticas, e num dos 2 pares (Luther Vandross) um dos títulos diz "Remastered", pelo que o que o md5 prova é que o excerto é o mesmo.

#### 4.1.3 A remix, an edit or a live version is the same song and a different recording. Is it a duplicate?

Depende da unidade de análise. Um remix, uma edição ou uma versão ao vivo é a mesma canção, mas outra gravação: pode partilhar artista, título e letra, mas o áudio é diferente.

- Como gravação, não é duplicado. Tem características acústicas próprias, logo é distinto e não se remove.
- Como canção, as linhas não são independentes. Partilham letra e melodia e, em geral, a emoção percebida.

#### 4.1.4 Does any recording, or any song, appear on both sides of the test boundary? What does that do to the test score, and what can be done about it without changing the test partition?

In [ ]:
def atravessam(c):
    d = u.dropna(subset=[c])
    if c == "AllMusic Id":
        d = d[d[c] != "not_found"] # porque sentinela não é um identificador
    g = d.groupby(c)["teste"].agg(["min", "max"])
    return g[(g["min"] == False) & (g["max"] == True)].index

test = pd.read_csv(TABLES / "test_songs.csv")
u["teste"] = u["Audio_Song"].isin(set(test["Audio_Song"]))
print("teste:", u["teste"].sum(), "| fora do teste:", (~u["teste"]).sum()) 

for i in ["audio", "AllMusic Id", "letra", "art_tit"]:
    chaves = atravessam(i)
    n = (u[i].isin(chaves) & u["teste"]).sum()
    print(i, "| músicas de teste afetadas:", n)

for i in ["audio", "letra"]:
    print(i)
    print(u[u[i].isin(atravessam(i))][["Audio_Song", "Artist", "Title", "Duration", "Year", "teste"]].sort_values(["Artist", "Title"]))

afetadas = set()
for i in ["audio", "letra"]:
    afetadas |= set(u[u[i].isin(atravessam(i)) & u["teste"]]["Audio_Song"])
print("músicas de teste afetadas (áudio ou letra):", len(afetadas))

teste: 409 | fora do teste: 914
audio | músicas de teste afetadas: 1
AllMusic Id | músicas de teste afetadas: 0
letra | músicas de teste afetadas: 3
art_tit | músicas de teste afetadas: 0
audio
        Audio_Song    Artist               Title  Duration        Year  teste
165   MT0003503243  Godsmack  I F****** Hate You     247.0  08/04/2003  False
1058          A028  Godsmack  I Fucking Hate You      -1.0         NaN   True
letra
        Audio_Song                          Artist  \
165   MT0003503243                        Godsmack   
1058          A028                        Godsmack   
998   MT0015665796                  Hank  Williams   
1118  MT0006540794                    Hank Ballard   
521   MT0005540907  Hank Ballard & the Midnighters   
155   MT0001134958                   Hank Williams   

                             Title  Duration        Year  teste  
165             I F****** Hate You     247.0  08/04/2003  False  
1058            I Fucking Hate You      -1.0         Na

Sim, poucos. Uma gravação aparece dos dois lados (áudio idêntico): Godsmack, "I F****** Hate You" no treino e "I Fucking Hate You" no teste. Ao nível da canção (letra idêntica) há 3 grupos que atravessam a fronteira: o dos Godsmack, Hank Williams ("I'm So Lonesome, I Could Cry" no teste e "I'm So Lonesome" no treino) e "Look at Little Sister" (Hank Ballard no teste, Hank Ballard & the Midnighters no treino). No total, 3 das 409 músicas de teste têm um par no treino. O artista+título exato não apanha nenhum, porque o título ou o artista estão escritos de forma diferente, e o `AllMusic Id` só repete o valor not_found, que não é um identificador.

O kNN encontra um vizinho quase igual e acerta por memorização, por isso a estimativa fica otimista. Com os casos detetados, efeito é pequeno: no máximo 3/409 ≈ 0,73% das músicas de teste. Podem existir mais casos, porque o md5 e a letra exata só apanham cópias idênticas.

O que se pode fazer sem mudar o teste:

- Retirar do treino as 3 linhas que têm par no teste, ou reportar a pontuação do teste com e sem as 3 músicas afetadas (409 e 406).
- Agrupar por canção na validação cruzada.

#### 4.1.5 How many artists are written in more than one way? What does that do to a cross-validation that places all the songs of each artist in the same fold?

In [16]:
import re, unicodedata

def limpar(nome):
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()   # tira acentos
    nome = re.sub(r"[^a-z0-9]+", " ", nome.lower()).strip()                          # minúsculas, sem pontuação nem espaços a mais
    return nome.removeprefix("the ")                                                  # tira o "the " do início

a = u.dropna(subset=["Artist"]).copy()
a["artista_norm"] = a["Artist"].map(limpar)

print("grafias distintas:", a["Artist"].nunique())
print("artistas depois de normalizar:", a["artista_norm"].nunique())

grafias = a.groupby("artista_norm")["Artist"].nunique()
print("artistas escritos de mais de uma forma:", (grafias > 1).sum())

grafias distintas: 835
artistas depois de normalizar: 777
artistas escritos de mais de uma forma: 58


A coluna Artist tem 835 grafias distintas, que passam a 777 artistas depois de normalizar maiúsculas, espaços, pontuação e o "The" inicial. Há 58 artistas escritos de mais de uma forma (todos com duas grafias), sempre por diferenças de maiúsculas, espaços a mais ou no fim, ou "The": por exemplo "ACE HOOD" e "Ace Hood", ou "Hank Williams" e "Hank  Williams". Há ainda pares que esta limpeza não junta, como "Cure" e "Cure, The" ou "Hank Ballard" e "Hank Ballard & the Midnighters", por isso 58 é um mínimo.

Esta validação usa Artist como grupo e coloca todas as músicas de cada grupo no mesmo fold. Se o mesmo artista tem duas grafias, o código vê dois artistas e pode pô-los em folds diferentes. Músicas do mesmo artista ficam então dos dois lados da divisão, a validação deixa de ser disjunta por artista e a estimativa fica otimista. Por isso o grupo usado é o nome normalizado.

#### 4.1.6 What should be done with each kind of duplicate? Justify

- Cópias exatas: mantém-se uma linha por par e removem-se as outras. São o mesmo registo, e deixá-las sobrepesa essas músicas. 
- Mesma gravação com ids diferentes: no par que atravessa a fronteira (Godsmack), mantém-se a linha do teste e remove-se a do treino, porque o teste não se altera e o kNN acertaria por memorização. No outro par (Luther Vandross), se estiver do mesmo lado, mantém-se uma linha, a mais completa, porque o áudio é idêntico e contaria duas vezes.
- Mesma canção, outra gravação: não se removem, porque o áudio é diferente e cada gravação tem características próprias. Ficam do mesmo lado de qualquer partição, agrupadas por canção na validação cruzada. Quando atravessam a fronteira (Hank Williams e Hank Ballard), a linha do treino sai, pelo mesmo motivo do Godsmack. 
- Total retirado do treino por fuga para o teste: 3 linhas. Nada se remove do teste.
- Artistas escritos de formas diferentes: cria-se um identificador a partir do nome normalizado, sem apagar a coluna original, e usa-se como grupo na validação por artista. 